In [1]:
import sys
import git
import pathlib

# Set up the PROJ_ROOT variable
PROJ_ROOT_PATH = pathlib.Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
PROJ_ROOT =  str(PROJ_ROOT_PATH)
if PROJ_ROOT not in sys.path:
    sys.path.append(PROJ_ROOT)

# Explicitly add the current notebook's directory
CURRENT_DIR = str(pathlib.Path().absolute())
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors
%matplotlib inline  

import yaml

In [3]:
from library.fridges import TEMP_STAGES, FRIDGE_LIBRARY
FRIDGE_NAME = "KIDE"
fridge = "kide"
cooling_power_budget = FRIDGE_LIBRARY[FRIDGE_NAME]["cooling_power"]
temp_stage = "4K"
QUBIT_GROUP_SIZE = 8

COOLING_POWER = cooling_power_budget[temp_stage]

In [4]:
def get_amp_hl(fridge, amp, wire):
    exp = "ff-fc-delft-"+ amp + "_" + wire + "-" + fridge
    fname = PROJ_ROOT_PATH / "notebooks" / "experiments" / "solutions" /fridge / amp / exp / f"{exp}.pkl"
    pqfname =  PROJ_ROOT_PATH / "notebooks" / "experiments" / "solutions" / fridge / amp / exp  / f"PQ_{exp}.pkl"
    df =  pd.read_pickle(fname)
    df_pq =  pd.read_pickle(pqfname)
    pq = df_pq.iloc[0].tolist()

    s = df.loc["4K", "AMP_BIAS"]

    phl = s[("PASSIVE", "IDLE")]
    ahl = s[("AMP", "IDLE")]
    if ("AMP_OHMIC", "IDLE") in s.index:
        ohl = s[("AMP_OHMIC", "IDLE")]
    else:
        ohl = 0.0

    NO_OF_QUBIT_GROUPS = int(min(pq)/QUBIT_GROUP_SIZE)

    # Factor of 2 because we need two amplifiers per qubit group
    scale = COOLING_POWER / NO_OF_QUBIT_GROUPS / 2
    phl *= scale
    ahl *= scale
    ohl *= scale

    thl = phl + ahl + ohl

    return phl, ahl, ohl, thl

In [5]:
_SI_PREFIX = {
    -24: "y", -21: "z", -18: "a", -15: "f", -12: "p", -9: "n",
    -6: "u", -3: "m", 0: "", 3: "k", 6: "M", 9: "G", 12: "T",
    15: "P", 18: "E", 21: "Z", 24: "Y",
}

def fmt_eng(x: float, sig: int = 3, unit: str = "W") -> str:
    """Engineering notation with SI prefix; exponent divisible by 3."""
    x = float(x)
    if x == 0.0 or not np.isfinite(x):
        return f"{x:g}{unit}"

    exp3 = int(np.floor(np.log10(abs(x)) / 3) * 3)
    exp3 = max(min(exp3, 24), -24)  # clamp to known prefixes
    scaled = x / (10 ** exp3)
    prefix = _SI_PREFIX[exp3]

    # sig significant figures, no trailing clutter
    s = f"{scaled:.2f}".rstrip('0').rstrip('.')
    # return rf"\qty{{{s}}}{prefix}{unit}".strip()
    return f"{s} {prefix}{unit}".strip()

In [6]:
amps  = ["hemt", "hemt-lp", "hemt-ulp", "sisv1"]
wires = ["cu", "mn", "ybco"]

data = {}
for amp in amps:
    row = {}
    for wire in wires:
        phl, ahl, ohl, thl = get_amp_hl(fridge, amp, wire)

        # Use \n instead of \newline
        row[wire] = (
            f"AHL: \t {fmt_eng(ahl)}\n"
            f"PHL: \t {fmt_eng(phl)}\n"
            f"OHL: \t {fmt_eng(ohl)}\n"
            f"THL: \t {fmt_eng(thl)}"
        )
    data[amp] = row

df_amp = pd.DataFrame.from_dict(data, orient="index")
df_amp.index.name = "Amplifier"
df_amp.columns.name = "Wire"

In [7]:
with pd.option_context('display.max_colwidth', None):
    # This CSS property forces the HTML container to respect the \n characters
    display(df_amp.style.set_properties(**{'white-space': 'pre-wrap'}))

Wire,cu,mn,ybco
Amplifier,,,
hemt,AHL: 7.8 mW PHL: 20.42 mW OHL: 390.76 nW THL: 28.22 mW,AHL: 7.8 mW PHL: 76.41 uW OHL: 467.63 uW THL: 8.34 mW,AHL: 7.8 mW PHL: 11.52 uW OHL: 0W THL: 7.81 mW
hemt-lp,AHL: 300 uW PHL: 20.42 mW OHL: 20.81 nW THL: 20.72 mW,AHL: 300 uW PHL: 76.41 uW OHL: 24.9 uW THL: 401.31 uW,AHL: 300 uW PHL: 11.52 uW OHL: 0W THL: 311.52 uW
hemt-ulp,AHL: 200 uW PHL: 40.85 mW OHL: 14.45 nW THL: 41.05 mW,AHL: 200 uW PHL: 152.82 uW OHL: 17.29 uW THL: 370.11 uW,AHL: 200 uW PHL: 23.05 uW OHL: 0W THL: 223.05 uW
sisv1,AHL: 8.1 uW PHL: 34.04 mW OHL: 1.12 uW THL: 34.05 mW,AHL: 8.1 uW PHL: 127.35 uW OHL: 1.34 mW THL: 1.47 mW,AHL: 8.1 uW PHL: 19.21 uW OHL: 0W THL: 27.31 uW
